# Ayudantía 12: IG 2 💻

## Ayudantes 👾

- S1: Josefa Buch
  - [IZ*ONE - DAYDREAM](https://open.spotify.com/track/4qj6sQ5Y3tY4MIEEzqGLhT?si=ZOvY8Mt4TpOX0LELdN_6jQ&utm_source=copy-link)
- S2: Bastián Pérez
  - [Daft Punk - Aerodynamic](https://open.spotify.com/track/3H3cOQ6LBLSvmcaV7QkZEu)
- S3: Carlos Martel
  - [Chystemc - Mi Fe](https://open.spotify.com/track/6eP769s7qMxoxKe2pWyyDf)
- S4: André Angulo
  - [Canserbero - C'est La Mort](https://open.spotify.com/track/5j2P3w4A2urLfFldFCmYLZ?si=wgkZfIZ4Tt2p1iJC8ZahfA&utm_source=native-share-menu)
- S5: Juan Pablo Olivares
  - [Álvaro Díaz - Ramona Flowers](https://open.spotify.com/track/6cOjQjg8GRhLozpoyiqvfp?si=_g0W_kFjSZqrRfnaauD1vQ&utm_source=copy-link)



## Contenidos 📖

- **Regla de oro:** solo el *thread* principal toca la interfaz; los demás *threads* le avisan mediante señales.
- ***Threads* en PyQt:** `threading.Thread` o `QThread` para lo que bloquea, y `QTimer` para lo que se repite y es rápido.
- **Integración entre interfaces gráficas y servidores:** un cliente con *front-end*, *back-end* y un *thread* que escucha al servidor.

## 🧵 IG 2 de un vistazo

Un mapa rápido de todo lo que vieron en los contenidos de la semana. Los ejercicios de hoy se concentran en los marcados con ✅.

| **Concepto** | **Qué es** | **Idea clave** | **Hoy** |
|:-------------|:-----------|:---------------|:--------|
| ***Event loop*** | Ciclo del *thread* principal que saca un evento de la cola y ejecuta su *slot* | Mientras un *slot* corre, el *event loop* está detenido: todo *slot* debe terminar rápido | ✅ Partes 1 y 2 |
| **Regla de oro** | Solo el *thread* principal modifica *widgets* | Los demás *threads* emiten señales, que se encolan y las ejecuta el *event loop* | ✅ Partes 1 y 2 |
| `QTimer` | Ejecuta un método cada cierto intervalo, dentro del *event loop* | No crea *threads*: sirve para lo periódico y rápido (un reloj, mover personajes) | — |
| `QThread` | La clase de *thread* de PyQt; se usa igual que `threading.Thread` | Puede definir sus propias señales. Solo `run` se ejecuta en el *thread* nuevo | 👀 Al final |
| `QMutex` | La versión de PyQt del `Lock` | `lock()` y `unlock()`, este último dentro de un `finally` | 👀 Al final |
| **Cierre limpio** | Detener *threads* y *timers* al cerrar una ventana o la aplicación | Bandera + `wait()` / `join()`, o `stop()` en un `QTimer` | — |
| **GUI y *networking*** | Cliente con *front-end*, *back-end* y un *thread* que escucha | `recv` bloquea, así que va en otro *thread*; enviar se puede hacer desde el principal | ✅ Partes 1 y 2 |
| ***Hook* de excepciones** | `sys.excepthook = hook` | Evita que un error dentro de un *slot* cierre todo el programa | — |

## ¡Gato Multijugador! 😸

Luego de una productiva tarde estudiando los contenidos de la semana de Programación Avanzada, planeabas jugar con tu amigo Pepito, pero Steam está caído. Te encuentras aburrido en tu pieza, mirando al techo, cuando de pronto recuerdas: puedes usar un _socket_ para comunicarme con Pepito, e integrar una interfaz gráfica para crear un juego! Así es como te dispones a la tarea de crear un juego clásico en multijugador: el **gato**.

# Parte 1

Tienes programada gran parte del juego. Tienes un servidor, y un cliente con una interfaz gráfica separada entre frontend y backend. Sin embargo, te encuentras con un error muy extraño: cuando se unen dos jugadores, el segundo jugador no pasa de la ventana de inicio a la de juego como debería. Deberás analizar el código y completar las partes correspondientes para arreglar este error.

Hint: recuerda que estás trabajando con _threads_. ¿Cómo sabe el servidor que el cliente ya conectó las señales correspondientes a la interfaz gráfica?

## 🚦 Resumen esencial de *threads* y señales

Una señal es la forma en que un *thread* secundario le avisa algo a la interfaz sin tocarla. El método conectado **no** se ejecuta en el *thread* que emite: la llamada se encola y la ejecuta el *event loop* del *thread* principal.

| **Categoría** | **Comando / Sintaxis** | **Descripción breve** | **Ejemplo** |
|:--------------|:-----------------------|:----------------------|:------------|
| **Definir** | `senal = pyqtSignal(tipos)` como atributo de clase | Solo en clases que heredan de `QObject`, no en un `threading.Thread` | `senal_turno = pyqtSignal(str)` |
| **Conectar** | `objeto.senal.connect(metodo)` | Desde ese momento, cada `emit` ejecuta `metodo` | `cliente.senal_inicio.connect(ventana_juego.show)` |
| **Emitir** | `objeto.senal.emit(args)` | Ejecuta los métodos conectados con esos argumentos | `self.senal_jugador.emit(self.simbolo)` |
| **Emitir desde otro *thread*** | `emit` dentro del *thread* secundario | La llamada se encola en el *thread* principal, que es el único que modifica la ventana | `procesar_mensaje` corre en `listen_thread` y emite `senal_inicio` |
| ***Thread* que escucha** | `threading.Thread(target=f, daemon=True)` | `recv` bloquea: en el *thread* principal congelaría la interfaz | `self.listen()` en `Cliente.__init__` |
| **Esperar a otro *thread*** | `evento = threading.Event()` | `evento.wait()` pausa el *thread* hasta que otro llame a `evento.set()` | `evento.wait()` |

> ⚠️ `emit` no guarda nada para después: si en ese momento no hay ningún método conectado a la señal, no pasa nada.

# Parte 2

Ahora, te falta programar los turnos. De momento, tienes que el apretar una casilla está conectado al método `Cliente.jugar`. Tienes que lograr una comunicación con el servidor de la siguiente forma:

- `Cliente.jugar` debería informarle al servidor de la casilla apretada, sólo si es el turno del cliente y la casilla no está ocupada. Si logra jugar, deja de ser su turno.

- `Server.procesar_mensaje` procesará este mensaje, y usará el método `Partida.procesar_jugada` para ejecutar la jugada. Debe informarle del resultado de esto a ambos jugadores, y en caso de tener un ganador o un empate, eliminar la partida usando `Server.eliminar_partida`

- De vuelta en el cliente, `Cliente.procesar_mensaje` deberá primero revisar si se mandó un nuevo tablero, y actualizarlo si es el caso. Para eso deberás completar la función `Cliente.actualizar_tablero`.

- También en `Cliente.procesar_mensaje`, si la acción del mensaje es `"turno"`, significa que se ha cambiado de turno. Deberás actualizar el _label_ correspondiente y cambiar el atributo que representa al turno.

## 🛜 Resumen esencial de GUI y *networking*

El cliente queda dividido en tres: la ventana (*front-end*), que no sabe nada de *sockets*; la lógica (*back-end*), que envía y procesa mensajes; y el *thread* que escucha al servidor. Los mensajes son diccionarios serializados con JSON, precedidos por 4 *bytes* con su largo.

| **Categoría** | **Comando / Sintaxis** | **Descripción breve** | **Ejemplo** |
|:--------------|:-----------------------|:----------------------|:------------|
| **Enviar** | `send_json(dict)` | Serializa con JSON y manda primero el largo (4 *bytes*) y después el mensaje | `self.send_json(mensaje)` |
| **Recibir** | `recv(4)` → `int.from_bytes(..., byteorder='big')` → `recv(largo)` | Primero se lee el largo y después el mensaje | `listen_thread` / `client_connection` |
| **Decidir qué hacer** | `msg["accion"]` | La llave `"accion"` dice qué tipo de mensaje llegó | `if msg["accion"] == "ganador":` |
| **Llave opcional** | `msg.get(llave)` | Retorna `None` si la llave no viene, en vez de lanzar `KeyError` | `msg.get("accion")` |
| **Servidor → un cliente** | `send_json(socket, dict)` | Le responde solo a ese cliente | `self.send_json(self.client_sockets[addr_1], {...})` |
| **Servidor → la partida** | `broadcast(partida, dict)` | Le manda el mismo mensaje a los dos jugadores | `self.broadcast(partida, {"accion": "turno", "simbolo": "X"})` |
| ***Back-end* → *front-end*** | `self.senal.emit(...)` | El *back-end* nunca toca *widgets*: avisa y la ventana se actualiza | `self.senal_fin.emit(self.simbolo, "ganaste")` |
| ***Front-end* → *back-end*** | `ventana.senal.connect(cliente.metodo)` | La ventana avisa qué hizo el usuario, sin saber nada del servidor | `ventana_juego.senal_click_label.connect(cliente.jugar)` |

> 💡 Enviar es rápido, así que se puede hacer directo desde el *thread* principal (por ejemplo, desde un método conectado a un *click*). Recibir bloquea, y por eso vive en `listen_thread`.

## 🧶 Resumen esencial de `QThread` y `QMutex`

PyQt trae sus propias versiones del `Thread` y del `Lock` de la semana de *threading*. Se usan casi igual; la diferencia es que, al ser parte de PyQt, se integran con el resto del *framework*. Ambas vienen en `PyQt6.QtCore`.

### `threading.Thread` vs. `QThread`

| | **`threading.Thread`** | **`QThread`** |
|:--|:-----------------------|:--------------|
| **Crear** | Heredar y definir `run`, o `Thread(target=f)` | Heredar y definir `run` |
| **Iniciar** | `start()` | `start()` |
| **Esperar a que termine** | `join()` | `wait()` |
| **¿Sigue corriendo?** | `is_alive()` | `isRunning()` |
| **Señales propias** | No: solo un `QObject` puede definirlas, así que hay que pasarle una desde afuera | Sí: `pyqtSignal` como atributo de clase |
| **Aviso al terminar** | — | Señal `finished`, cuando `run` termina |
| ***Daemon*** | `daemon=True` | No existe: se detiene con una bandera que `run` revisa, y después `wait()` |

```python
from time import sleep
from PyQt6.QtCore import QThread, pyqtSignal


class Contador(QThread):
    senal_numero = pyqtSignal(str)  # la señal es del propio thread

    def run(self):
        for i in range(10):
            sleep(0.5)
            self.senal_numero.emit(str(i))


# Dentro de la ventana
self.contador = Contador()  # en un atributo, no en una variable local
self.contador.senal_numero.connect(self.label.setText)
self.contador.start()
```

Con `threading.Thread` el `run` es el mismo, pero la señal tiene que venir de afuera, de un `QObject` como la ventana:

```python
class Contador(Thread):
    def __init__(self, senal_numero):
        super().__init__()
        self.senal_numero = senal_numero  # señal definida en la ventana
```

### `threading.Lock` vs. `QMutex`

| | **`threading.Lock`** | **`QMutex`** |
|:--|:---------------------|:-------------|
| **Crear** | `lock = Lock()` | `mutex = QMutex()` |
| **Tomar** | `lock.acquire()` | `mutex.lock()` |
| **Soltar** | `lock.release()` | `mutex.unlock()` |
| **Forma segura** | `with lock:` | `lock()` y después `try` / `finally: unlock()` |
| **¿Sirve con `QThread`?** | Sí, funciona igual | Sí |

```python
from PyQt6.QtCore import QMutex, QThread


class Deposito(QThread):
    def __init__(self, cuenta, mutex):
        super().__init__()
        self.cuenta = cuenta
        self.mutex = mutex  # el MISMO mutex para todos los threads

    def run(self):
        for _ in range(100_000):
            self.mutex.lock()
            try:
                self.cuenta.saldo += 1  # sección crítica
            finally:
                self.mutex.unlock()


mutex = QMutex()
depositos = [Deposito(cuenta, mutex) for _ in range(4)]
```

Con un `Lock`, el `run` queda así:

```python
    def run(self):
        for _ in range(100_000):
            with self.lock:
                self.cuenta.saldo += 1
```

> ⚠️ **Solo `run` se ejecuta en el *thread* nuevo.** Cualquier otro método del `QThread` conectado a una señal se ejecuta en el *thread* principal y congela la ventana: el trabajo pesado va dentro de `run`.

> ⚠️ **Guarda siempre la referencia** al `QThread` mientras corre (en un atributo o una lista). Si queda solo en una variable local, Python lo elimina y el programa se cae con `QThread: Destroyed while thread is still running`. Tampoco lo guardes en `self.thread`: todo objeto de PyQt ya tiene un método `thread()`.

> 💡 Si respetas la regla de oro y los *threads* solo se comunican con señales, casi todo el estado se modifica en el *thread* principal, y rara vez vas a necesitar un *lock* o un *mutex*.